# TFM_NAS — CP 3.4 TPE resume + anchor-B on Colab T4

A thin driver over the committed `colab/` scripts. **Runtime → Change runtime type → T4 GPU** first.

**One-time, into your Drive** (`MyDrive/tfm_nas/`):
- `secrets/access_token` + `secrets/kaggle_username` — upload your repo `secrets/` files (Kaggle CLI auth).
- `cache/cp33_bo_cache_r640.*.jsonl` — upload your local `data/cp33_kaggle_out/cp33_bo_cache_r640.*.jsonl` so the TPE DoD **resumes** (seeds 0/1 done, 2–4 partway) instead of restarting. (Or pass `--seed-cache` to pull the Kaggle cache Dataset instead.)

The 1.6 GB gate dataset is **not** uploaded by hand — it is pulled from the existing `tfm-nas-gate-pose` Kaggle Dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/tfm_nas'

In [ ]:
import os
REPO = '/content/TFM_NAS'
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Asilon47/TFM_NAS.git {REPO}
else:
    !cd {REPO} && git pull --ff-only

## (Optional) Anchor-B accuracy — yolo11s-pose gate fine-tune on the T4
~1 h, resumable via Drive. **Off** the CP 3.5 critical path (feeds only the λ robustness check + Phase-8 scouting). Run it to reclaim your laptop CPU, or skip it — it competes with the TPE run for the single T4.

In [ ]:
!python {REPO}/colab/anchor_b.py --drive {DRIVE_ROOT}

## CP 3.4 — TPE @640 DoD (the critical one)
Resumes seeds 2–4 from the Drive cache; every eval persists to Drive as it lands. If Colab disconnects, just **re-run this cell** — it continues. Repeat until the log prints `complete=true`.

In [ ]:
!python {REPO}/colab/run_colab.py --drive {DRIVE_ROOT} --seed-cache

## When it prints `complete=true`
Outputs are on Drive under `MyDrive/tfm_nas/out/`:
- `cp34_tpe.json` — the TPE-vs-random DoD verdict
- `anchor_yolo11s_pose_640_map.json` — anchor-B accuracy (if you ran it)

Download both into the laptop repo's `data/`, then pick the winner over the BO∪TPE union:
```bash
python -m search.select_winner --frontier data/cp33_bo.json data/cp34_tpe.json
```